[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/%E6%A9%9F%E7%8E%87%E7%B5%B1%E8%A8%88/03_%E4%BA%8C%E9%A0%85%E5%88%86%E4%BD%88.ipynb) · [回章節首頁](./README.md)

## 這個觀念要幫我們回答什麼？

**核心問題：罰球命中率 8 成、罰 5 球，「剛好進 3 球」的機率是多少？**

- **用在流程哪裡**：把一批二元結果計數時的機率計算（品管、投籃、批量預測的理想化模型）。
- **讀完要能做到**：能檢查二項假設是否成立，會算剛好、至多、至少的成功次數。
- **閱讀方式**：每個程式格前先讀「為什麼做／怎麼看」，再執行；用實際輸出回答「先想一想」。

觀念連結：[數據結構](../名詞解釋/02_數據結構/README.md)、[評估指標](../名詞解釋/09_評估指標/README.md)

# 03｜二項分布：重複試驗後成功幾次？

二項分布回答：「**固定做 n 次，剛好成功 k 次**的機率是多少？」

**一、先檢查四個條件（缺一不可）**

1. **固定次數 n**：一開始就說好做幾次（罰 5 球），不能中途加碼。
2. **每次只有兩種結果**：成功／失敗（進／不進、良品／瑕疵）。
3. **成功率 p 每次相同**：每次都是 0.8，不會越投越準。
4. **各次獨立**：上一球進不進，不影響下一球。

真實投籃有疲勞、手感、防守，條件 3、4 常常不成立——本篇是**理想化模型**，先學會算，再學會懷疑。

**二、公式與數字感**

- **PMF**：$P(X=k) = C(n,k)\, p^k (1-p)^{n-k}$，$k = 0,1,...,n$。$C(5,3)=10$ 種「哪 3 球進」的組合，每種機率 $0.8^3 \times 0.2^2$，相乘得 $20.48\%$。
- **期望與變異**：$E[X] = np$（5 球期望進 4 球；期望可以不是整數，它是長期平均，不是每次的預言）、$Var(X) = np(1-p)$。
- **累積怎麼算**：「至多 3 球」用 CDF $P(X \le 3)$；「至少 4 球」用存活函數 $P(X \ge 4) = sf(3)$，即 $1 - CDF(3)$。注意 `sf(3)` 的 3 是「扣除的分界」，不是目標本身。
- **形狀直覺**：$p < 0.5$ 中心靠左、右尾長（右偏）；$p > 0.5$ 反之。只有 $p=0.5$ 長得像對稱鐘形，**不是所有二項分布都像常態**。

**三、三個字母分工**：$n$ 是每組做幾次，$p$ 是每次成功率，`size` 是做幾組（模擬 2000 組罰球，不是把 n 改成 2000）。每組結果是 0～5 的一個數，2000 組的比例會圍繞理論 PMF 波動。

把一批分類結果當二項計數前，必須先檢查獨立性與相同成功率；不同族群、不同難度的樣本會違反假設，也不要把模型輸出分數直接代入 $p$ 當真實成功率。

## 準備共用繪圖設定

**為什麼做**：統一繁體中文字型與配色，讓讀圖重點清楚。

**怎麼看**：這格只載入工具，沒有訓練模型；會列出實際字型。需要套件時由老師協助準備 NumPy、pandas、SciPy、Matplotlib、Seaborn。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

# 設定中文字型支援
# --- 設定 Matplotlib 中文字型支援（Mac / Windows / Colab 通用）---
import glob
import os
try:
    import mplfonts
except ImportError:  # Colab 環境若缺套件則自動安裝（本機已在 requirements.txt）
    %pip install -q mplfonts
    import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)   # 直接註冊 mplfonts 內附的字型，不改任何檔案
use_font("Noto Sans CJK SC")              # 套用思源黑體，負號也會一起處理

sns.set_theme(style='whitegrid', rc={
    'axes.unicode_minus': False, 'figure.facecolor': '#F5F7F2',
    'axes.facecolor': '#FFFFFF', 'axes.spines.top': False,
    'axes.spines.right': False, 'grid.alpha': 0.22,
    'axes.titlesize': 14, 'axes.labelsize': 11, 'figure.dpi': 110})
GREEN, ORANGE, PURPLE = '#2F6657', '#C17B45', '#786497'
rng = np.random.default_rng(42)
print('中文字型：Noto Sans CJK SC（mplfonts 通用寫法）')

## 01｜先檢查這是不是二項問題

**為什麼做**：固定 n 次、每次二元結果、相同成功率 p，而且各次獨立。這裡假設五次罰球成功率均為 0.8。

**怎麼看**：橫軸 k 是成功次數，n 是每組試驗次數；不是每根柱有 n 筆資料。真實投籃可能受疲勞或情境影響，本例是理想化模型。

In [ ]:
n, p = 5, 0.8
k = np.arange(n + 1)
probs = stats.binom.pmf(k, n, p)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(k, probs, color=[ORANGE if i == 3 else GREEN for i in k])
ax.set(xticks=k, xlabel='五次中投進的次數', ylabel='機率',
       title='剛好三次成功：看 k＝3 那根柱')
plt.show()
print(f'剛好投進 3 次：{stats.binom.pmf(3, n, p):.2%}')
print(f'至少投進 4 次：{stats.binom.sf(3, n, p):.2%}')
print(f'至多投進 3 次：{stats.binom.cdf(3, n, p):.2%}')
assert np.isclose(probs.sum(), 1)
assert np.isclose(stats.binom.pmf(3, 5, .8), .2048)

## 02｜p 決定中心方向，n 決定計數範圍

**為什麼做**：一次比較一個參數，理解期望成功次數 np；期望可以不是整數，不是每次必然出現的結果。

**怎麼看**：p 小於 0.5 時中心靠左、右尾較長，稱右偏；p 大於 0.5 時中心靠右、左尾較長，稱左偏。不是所有二項分布都像鐘形。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7), layout='constrained')
for ax, (trials, chance) in zip(axes.flat, [(10, .2), (10, .5), (10, .8), (30, .8)]):
    values = np.arange(trials + 1)
    ax.bar(values, stats.binom.pmf(values, trials, chance), color=GREEN)
    ax.axvline(trials * chance, color=ORANGE, linestyle='--', label=f'期望：{trials * chance:g}')
    ax.set(title=f'試驗 {trials} 次／成功率 {chance:.0%}',
           xlabel='成功次數', ylabel='機率')
    ax.legend()
plt.show()

## 03｜size 是做幾組，不是每組做幾次

**為什麼做**：模擬很多組五次罰球，讓理論 PMF 與實際比例對照。

**怎麼看**：每一個抽樣值介於 0～5，表示一組五次中成功幾次。size=2000 表示 2,000 組，不是把 n 改成 2,000。

In [ ]:
successes = np.random.default_rng(42).binomial(n=5, p=.8, size=2000)
observed = np.bincount(successes, minlength=6) / len(successes)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(k - .18, probs, width=.36, color=GREEN, label='理論 PMF')
ax.bar(k + .18, observed, width=.36, color=ORANGE, label='2,000 組的觀察比例')
ax.set(xticks=k, title='有限次抽樣會有波動', xlabel='成功次數', ylabel='機率／比例')
ax.legend()
plt.show()

## 接回機器學習

把一批分類結果當作二項計數，必須先檢查是否獨立、是否有相同成功率。不同人群或不同難度的樣本可能違反假設；不要直接把模型分數代入 p 當成真實成功率。

## 換你解讀

1. 「至少 4 次」為什麼可以用 sf(3)？
2. 右偏指中心靠右，還是右邊的尾巴比較長？
3. n、p、size 分別改變什麼？

### 請 AI 幫你做一個小實驗

請 AI 固定 n=10，比較 p=0.1、0.5、0.9。先預測中心與長尾方向，再用圖驗證。

先說出預期，再看實際圖形，最後寫下「證據 → 解釋 → 不能下的結論」。

[回章節首頁](./README.md)